# Day 13 — Rolling Portfolio Optimization

## Objective
Build a rolling-window portfolio backtest using five stocks:
AAPL, BLK, GS, JPM and MSFT.

## Methodology
- 60-trading-day historical estimation window
- Monthly portfolio rebalancing
- Equal-weight, minimum-variance and risk-parity strategies
- Transaction-cost sensitivity
- Out-of-sample performance evaluation

## Key rule
Portfolio weights must be calculated using only information
available before the returns being evaluated.

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Load historical stock prices
prices = pd.read_csv(
    "five_stock_prices.csv",
    index_col=0,
    parse_dates=True
)

# Clean and organize the data
prices = prices.sort_index()
prices = prices[["AAPL", "BLK", "GS", "JPM", "MSFT"]]
prices = prices.dropna()

# Calculate daily returns
returns = prices.pct_change().dropna()

print("Stocks:", list(returns.columns))
print("First date:", returns.index.min().date())
print("Last date:", returns.index.max().date())
print("Trading days:", len(returns))
print("Missing values:", returns.isna().sum().sum())

display(returns.head())

Stocks: ['AAPL', 'BLK', 'GS', 'JPM', 'MSFT']
First date: 2025-09-24
Last date: 2026-09-21
Trading days: 249
Missing values: 0


,AAPL,BLK,GS,JPM,MSFT
Date,,,,,
2025-09-24,-0.008332,-0.003331,-0.017226,0.002174,0.001807
2025-09-25,0.018073,0.017119,0.002940,0.000096,-0.006116
2025-09-26,-0.005489,0.007984,0.009751,0.008327,0.008737
2025-09-29,-0.004032,0.016525,0.002006,-0.001171,0.006139
2025-09-30,0.000786,-0.008243,-0.009663,-0.000824,0.006510


In [2]:

# Day 13 — Step 3: Rolling windows and rebalance dates

lookback = 60

# First trading day of each month, after enough history exists
eligible_dates = returns.index[lookback:]

rebalance_dates = (
    pd.Series(eligible_dates, index=eligible_dates)
    .groupby(eligible_dates.to_period("M"))
    .first()
    .tolist()
)

# Inspect the first three rolling estimation windows
for date in rebalance_dates[:3]:
    position = returns.index.get_loc(date)

    # Use only the 60 trading days BEFORE the rebalance date
    window = returns.iloc[position - lookback:position]

    print(f"Rebalance date: {date.date()}")
    print(
        f"Estimation period: "
        f"{window.index.min().date()} to "
        f"{window.index.max().date()}"
    )
    print(f"Observations: {len(window)}")
    print("-" * 45)

print("Total scheduled rebalances:", len(rebalance_dates))

Rebalance date: 2025-12-18
Estimation period: 2025-09-24 to 2025-12-17
Observations: 60
---------------------------------------------
Rebalance date: 2026-01-02
Estimation period: 2025-10-07 to 2025-12-31
Observations: 60
---------------------------------------------
Rebalance date: 2026-02-02
Estimation period: 2025-11-04 to 2026-01-30
Observations: 60
---------------------------------------------
Total scheduled rebalances: 10


In [3]:

# Step 4: Calculate portfolio weights at each rebalance

n_assets = len(returns.columns)
equal_weights = np.ones(n_assets) / n_assets

def minimum_variance_weights(cov):
    def portfolio_variance(w):
        return w @ cov @ w

    result = minimize(
        portfolio_variance,
        equal_weights,
        method="SLSQP",
        bounds=[(0, 1)] * n_assets,
        constraints=[{
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1
        }],
        options={"ftol": 1e-12, "maxiter": 2000}
    )

    if not result.success:
        raise RuntimeError(result.message)

    return result.x


def risk_parity_weights(cov):
    def risk_contribution(w):
        volatility = np.sqrt(w @ cov @ w)
        return w * (cov @ w) / volatility

    def objective(w):
        contributions = risk_contribution(w)
        target = contributions.sum() / n_assets
        return np.sum((contributions - target) ** 2)

    result = minimize(
        objective,
        equal_weights,
        method="SLSQP",
        bounds=[(0.0001, 1)] * n_assets,
        constraints=[{
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1
        }],
        options={"ftol": 1e-12, "maxiter": 2000}
    )

    if not result.success:
        raise RuntimeError(result.message)

    return result.x


# Calculate weights using the preceding 60 trading days
rolling_weights = {}

for date in rebalance_dates:
    position = returns.index.get_loc(date)
    window = returns.iloc[position - lookback:position]

    cov = window.cov().values * 252

    rolling_weights[date] = {
        "Equal Weight": equal_weights.copy(),
        "Minimum Variance": minimum_variance_weights(cov),
        "Risk Parity": risk_parity_weights(cov)
    }

# Display allocations for the first rebalance
first_date = rebalance_dates[0]

allocation_table = pd.DataFrame(
    rolling_weights[first_date],
    index=returns.columns
) * 100

print("First rebalance:", first_date.date())
display(allocation_table.round(2))

print("Allocation totals (%):")
print(allocation_table.sum().round(2))


First rebalance: 2025-12-18


,Equal Weight,Minimum Variance,Risk Parity
AAPL,20.0,33.27,24.75
BLK,20.0,13.11,17.42
GS,20.0,0.00,13.50
JPM,20.0,17.87,17.83
MSFT,20.0,35.75,26.50


Allocation totals (%):
Equal Weight        100.0
Minimum Variance    100.0
Risk Parity         100.0
dtype: float64


In [5]:

# Step 5: Rolling backtest with transaction costs

initial_capital = 10_000
transaction_cost = 0.001  # 0.10% of traded value

strategies = [
    "Equal Weight",
    "Minimum Variance",
    "Risk Parity"
]

def run_backtest(strategy, cost_rate=0.001):
    dates = returns.loc[rebalance_dates[0]:].index

    # Start in cash before the first rebalance
    capital = initial_capital
    holdings = np.zeros(n_assets)
    portfolio_values = []
    turnover_records = []

    for date in dates:
        if date in rolling_weights:
            target_weights = rolling_weights[date][strategy]

            # Portfolio value before trading
            current_value = capital + holdings.sum()

            # Existing asset weights, excluding cash
            current_weights = holdings / current_value

            # Approximate turnover and trading costs
            turnover = np.abs(
                target_weights - current_weights
            ).sum()

            trading_cost = (
                current_value * turnover * cost_rate
            )

            # Reallocate after deducting costs
            capital = 0.0
            holdings = (
                current_value - trading_cost
            ) * target_weights

            turnover_records.append({
                "Date": date,
                "Turnover": turnover,
                "Trading Cost": trading_cost
            })

        # Apply the day's returns after rebalancing.
        # NOTE: This assumes trades occur before today's return.
        holdings *= 1 + returns.loc[date].values

        portfolio_values.append(holdings.sum())

    values = pd.Series(
        portfolio_values,
        index=dates,
        name=strategy
    )

    turnover_df = pd.DataFrame(turnover_records)

    return values, turnover_df


results = {}
turnover_results = {}

for strategy in strategies:
    values, turnover_df = run_backtest(
        strategy,
        transaction_cost
    )

    results[strategy] = values
    turnover_results[strategy] = turnover_df

portfolio_values = pd.DataFrame(results)

display(portfolio_values.tail().round(2))

print("\nFinal portfolio values:")
display(portfolio_values.iloc[-1].round(2))

print("\nTotal transaction costs:")
for strategy in strategies:
    total_cost = turnover_results[strategy][
        "Trading Cost"
    ].sum()

    print(f"{strategy}: ${total_cost:,.2f}")

,Equal Weight,Minimum Variance,Risk Parity
Date,,,
2026-09-15,11229.42,11412.64,11188.02
2026-09-16,11053.58,11326.65,11036.19
2026-09-17,11186.74,11396.56,11154.33
2026-09-18,11175.09,11394.22,11149.85
2026-09-21,11326.42,11491.39,11287.46



Final portfolio values:


Equal Weight        11326.42
Minimum Variance    11491.39
Risk Parity         11287.46
Name: 2026-09-21 00:00:00, dtype: float64


Total transaction costs:
Equal Weight: $13.84
Minimum Variance: $31.20
Risk Parity: $17.50


In [6]:
import numpy as np
import pandas as pd

daily_returns = portfolio_values.pct_change().dropna()

metrics = pd.DataFrame(index=portfolio_values.columns)

metrics["Total Return (%)"] = (
    portfolio_values.iloc[-1] / portfolio_values.iloc[0] - 1
) * 100

metrics["Annualized Volatility (%)"] = (
    daily_returns.std() * np.sqrt(252) * 100
)

metrics["Sharpe Ratio"] = (
    daily_returns.mean() * 252
    / (daily_returns.std() * np.sqrt(252))
)

running_peak = portfolio_values.cummax()
drawdown = portfolio_values / running_peak - 1

metrics["Max Drawdown (%)"] = drawdown.min() * 100

display(metrics.round(2))

,Total Return (%),Annualized Volatility (%),Sharpe Ratio,Max Drawdown (%)
Equal Weight,13.01,19.18,0.95,-15.43
Minimum Variance,14.43,17.42,1.12,-15.33
Risk Parity,12.52,18.34,0.95,-15.05


In [7]:
metrics["Total Return (%)"] = (
    portfolio_values.iloc[-1] / 10000 - 1
) * 100

display(metrics.round(2))

,Total Return (%),Annualized Volatility (%),Sharpe Ratio,Max Drawdown (%)
Equal Weight,13.26,19.18,0.95,-15.43
Minimum Variance,14.91,17.42,1.12,-15.33
Risk Parity,12.87,18.34,0.95,-15.05


### Day 13 — Research Conclusion

The rolling portfolio backtest compared equal-weight, minimum-variance and risk-parity allocations using historical data and periodic rebalancing.

Minimum variance generated a 14.91% total return with 17.42% annualized volatility and a Sharpe ratio of 1.12. Equal weight returned 13.26%, while risk parity returned 12.87%.

The results illustrate how portfolio construction methods can produce different return and risk profiles. They are specific to the historical sample and depend on estimation windows, rebalancing frequency and transaction-cost assumptions.

**Limitations:** A longer testing period, additional market regimes, realistic execution assumptions and sensitivity analysis are necessary before drawing conclusions about live investment performance.
